# Hückel model QPE with `qdk-chemistry`

This notebook demonstrates a complete workflow from deriving a [Hückel model](https://en.wikipedia.org/wiki/H%C3%BCckel_method) Hamiltonian for benzene's π-system to estimating its ground state energy using iterative Quantum Phase Estimation (iQPE). This is one example of a wide range of functionality in `qdk-chemistry`. Please see <https://github.com/microsoft/qdk-chemistry> for the full documentation.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need to install the `jupyter` extra to run this notebook:

```bash
pip install 'qdk-chemistry[jupyter]'
```

This installs the additional dependencies required by this notebook (ipykernel, pandas, numpy, and pyscf).

In [ ]:
# Load frequently used external packages
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## Deriving Hückel parameters from benzene

The Hückel model is a non-interacting tight-binding model containing only one-body integrals. Its Hamiltonian takes the form:

$$\hat{H}_\text{Hückel} = \sum_i \varepsilon\, \hat{n}_i \;-\; \sum_{\langle i,j \rangle} t\, (\hat{a}_i^\dagger \hat{a}_j + \text{h.c.})$$

Rather than choosing parameters by hand, we derive $\varepsilon$ and $t$ from a real benzene molecule using `qdk-chemistry`. The structure is loaded from an [XYZ-format](https://en.wikipedia.org/wiki/XYZ_file_format) file.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import create
from qdk.widgets import MoleculeViewer

# Load benzene structure from XYZ file
structure = Structure.from_xyz_file(Path("data/benzene.structure.xyz"))

# Visualize the benzene structure
MoleculeViewer(molecule_data=structure.to_xyz())

### Generating the molecular orbitals

This step performs a Hartree-Fock (HF) self-consistent field (SCF) calculation to generate an approximate initial wavefunction and ground-state energy guess. The minimal STO-3G basis set is used here for simplicity.

In [ ]:
# Run SCF
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g")
print(f"Hartree-Fock energy: {E_hf:.6f} Hartree")

### Identifying the π-orbitals

To parameterize the Hückel model, we need to identify the 6 carbon π-orbitals out of benzene's full set of 36 molecular orbitals (in STO-3G). This is a common task in quantum chemistry and `qdk-chemistry` provides automated tools to accomplish it.

The workflow uses three steps: reduce to the valence space, compute orbital entropies via SCI, and apply autoCAS-EOS to select the strongly correlated π-subspace. We first reduce the orbital space to the valence shell, removing the chemically inert core orbitals.

In [ ]:
from qdk_chemistry.utils import compute_valence_space_parameters

# Reduce to valence space
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
active_space_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o,
)
valence_wf = active_space_selector.run(wfn_hf)
print("Valence orbitals:\n", valence_wf.get_orbitals().get_summary())

Next, we compute a selected configuration interaction (SCI) wavefunction for the valence space. The SCI calculation provides single-orbital entropies that quantify how strongly correlated each orbital is.

In [ ]:
# Construct a Hamiltonian from the valence orbitals
hamiltonian_constructor = create("hamiltonian_constructor", "qdk_cholesky")
val_orbitals = valence_wf.get_orbitals()
val_hamiltonian = hamiltonian_constructor.run(val_orbitals)
num_alpha_electrons, num_beta_electrons = valence_wf.get_active_num_electrons()

# Compute the SCI wavefunction with RDMs for entropy analysis
macis_mc = create(
    "multi_configuration_calculator",
    "macis_asci",
    calculate_single_orbital_entropies=True,
)
_, wfn_sci = macis_mc.run(val_hamiltonian, num_alpha_electrons, num_beta_electrons)

With the orbital entropies in hand, we apply the `autoCAS-EOS` selector to automatically identify the strongly correlated subspace. For benzene, we expect it to select the 6 out-of-plane π-orbitals — a CAS(6,6) active space.

In [ ]:
# Optimize the problem with autoCAS-EOS active space selection
autocas = create("active_space_selector", "qdk_autocas_eos")
autocas_wfn = autocas.run(wfn_sci)
indices, _ = autocas_wfn.get_orbitals().get_active_space_indices()
n_active_e = sum(autocas_wfn.get_active_num_electrons())
print(f"autoCAS-EOS selected CAS({n_active_e},{len(indices)}) active space")
print(f"  Active orbital indices: {list(indices)}")

### Visualizing the selected orbitals

Visualizing the selected active space orbitals is an important step to verify that the selection is chemically meaningful. We first show the canonical (delocalized) orbitals selected by autoCAS.

In [ ]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Visualize the autoCAS-selected active space orbitals (canonical)
active_orbitals = autocas_wfn.get_orbitals()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=active_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

The canonical orbitals are delocalized over the ring. For the Hückel model we need site-localized orbitals, so we apply a Pipek-Mezey localization to produce one orbital per carbon atom, which will later represent a site in the model Hamiltonian.

In [ ]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Localize the active-space orbitals for clearer visualization
localizer = create("orbital_localizer", "qdk_pipek_mezey")
loc_wfn = localizer.run(autocas_wfn, *autocas_wfn.get_orbitals().get_active_space_indices())

# Visualize the localized active space orbitals
loc_orbitals = loc_wfn.get_orbitals()
loc_indices, _ = loc_orbitals.get_active_space_indices()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=loc_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=loc_indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

### Extracting Hückel parameters from the active space

With the localized π-orbitals in hand, we construct the active-space Hamiltonian and use its one-body integrals to extract the Hückel parameters without assuming any orbital ordering:
- **On-site energy** $\varepsilon$: average of the diagonal elements
- **Hopping integral** $t$: for each orbital, the two largest absolute off-diagonal elements identify its nearest neighbours in the ring

Note that the Pipek-Mezey localization does not preserve any particular ordering of the orbitals, so we cannot simply read off nearest-neighbour hoppings from adjacent matrix indices. Instead, we rely on the magnitude of the off-diagonal elements to identify the connections.

In [ ]:
n_sites = len(indices)
n_active_alpha, n_active_beta = autocas_wfn.get_active_num_electrons()

# Construct the active-space Hamiltonian from the localized orbitals
hamiltonian_constructor = create("hamiltonian_constructor")
refined_orbitals = loc_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)

# Get effective one-body integrals
h1_alpha, _ = active_hamiltonian.get_one_body_integrals()
print(f"Effective one-body integrals ({n_sites}x{n_sites}):\n{np.round(h1_alpha, 4)}")

# On-site energy: average diagonal element
epsilon = float(np.mean(np.diag(h1_alpha)))

# Hopping integral: each localized orbital has exactly 2 nearest neighbours in the ring.
# For each row, pick the 2 largest absolute off-diagonal elements and average.
nn_hoppings = []
for i in range(n_sites):
    row = np.abs(h1_alpha[i])
    row[i] = 0.0  # exclude diagonal
    top2 = np.sort(row)[-2:]
    nn_hoppings.extend(top2)
t = float(np.mean(nn_hoppings))

print(f"\nHückel parameters from CAS({n_active_alpha + n_active_beta},{n_sites}) active space:")
print(f"  On-site energy ε = {epsilon:.4f} Hartree")
print(f"  Hopping integral t = {t:.4f} Hartree")

### Building the Hückel Hamiltonian

Benzene's π-system has a ring topology, which we represent as a periodic 6-site chain. The Hückel Hamiltonian is constructed from the ab initio-derived parameters.

In [ ]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_huckel_hamiltonian

# Build ring lattice (periodic chain = benzene topology) and Hückel Hamiltonian
lattice = LatticeGraph.chain(n_sites, periodic=True)
hamiltonian = create_huckel_hamiltonian(lattice, epsilon=epsilon, t=t)

print(f"Hückel model: {lattice.num_sites} sites (ring), {n_active_alpha}α + {n_active_beta}β electrons")
print(f"One-body integrals (Hückel Hamiltonian):\n{np.round(hamiltonian.get_one_body_integrals()[0], 4)}")

## Classical reference energy

We solve the Hückel model exactly using the CASCI multi-configuration calculator to obtain a reference energy for benchmarking QPE. The configuration weights reveal the structure of the ground state.

In [ ]:
from qdk_chemistry.algorithms import create
from qdk.widgets import Histogram

# Exact diagonalization (CASCI)
mc_calculator = create("multi_configuration_calculator", "macis_cas")
e_exact, wfn_exact = mc_calculator.run(hamiltonian, n_active_alpha, n_active_beta)
print(f"Exact ground state energy: {e_exact:.6f} Hartree")

# Plot top configuration weights from the CASCI wavefunction
num_configurations = len(wfn_exact.get_active_determinants())
print(f"Total configurations in the CASCI wavefunction: {num_configurations}")
top_dets = wfn_exact.get_top_determinants()
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in top_dets.items()},
        items="top-25",
        sort="high-to-low",
    )
)

## Optimizing trial wavefunction loading onto a quantum computer

The multi-configuration wavefunction can serve as the trial state for the iQPE algorithm. However, this information needs to be loaded as a state on a quantum computer.

The amount of data loaded onto the quantum computer can be optimized by exploiting the sparsity of the wavefunction. We take the top determinants from the CASCI solution and check their overlap with the full wavefunction.

In [ ]:
from utils.qpe_utils import prepare_top_dets_trial_state

# Prepare a trial state from the top 2 determinants. Compute its overlap with the CASCI wavefunction.
wfn_trial, fidelity = prepare_top_dets_trial_state(wfn_exact, hamiltonian, num_dets=10)
print(f"Overlap of trial state with CASCI wavefunction: {fidelity:.2%}")

# Generate a plot of the configurations in the trial wavefunction
configurations = wfn_trial.get_top_determinants()
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in configurations.items()},
        sort="high-to-low",
    )
)

There are many ways to "load" this state onto a quantum computer. This example uses a popular method as a basis for comparison with our chemistry-aware optimized approach. Circuit statistics are shown and the circuit is visualized using built-in `qdk` functions.

In [ ]:
import qdk_chemistry.plugins.qiskit # Enable Qiskit plugin capabilities # noqa: F401
from qdk.openqasm import estimate
from qdk.widgets import Circuit

# Generate state preparation circuit for the sparse state using the regular isometry method (Qiskit)
state_prep = create("state_prep", "qiskit_regular_isometry")
regular_isometry_circuit = state_prep.run(wfn_trial)

# Visualize the regular isometry circuit
display(Circuit(regular_isometry_circuit.get_qsharp_circuit()))

# Print logical qubit counts estimated from the circuit
df = pd.DataFrame(
    estimate(regular_isometry_circuit.get_qasm()).logical_counts.items(),
    columns=['Logical Estimate', 'Counts']
)
display(df)

The popular approach for state preparation requires a larger number of operations with numerous fine rotations. However, `qdk-chemistry` provides optimized state preparation methods that exploit the structure of chemistry wavefunctions to reduce the number of operations and improve noise resilience.

In [ ]:
from qdk.openqasm import estimate
from qdk.widgets import Circuit

# Generate state preparation circuit for the sparse state via GF2+X sparse isometry
state_prep = create("state_prep", "sparse_isometry_gf2x")
state_prep_circuit = state_prep.run(wfn_trial)

# Visualize the sparse isometry circuit
display(Circuit(state_prep_circuit.get_qsharp_circuit()))

# Print logical qubit counts estimated from the circuit
df = pd.DataFrame(
    estimate(state_prep_circuit.get_qasm()).logical_counts.items(),
    columns=['Logical Estimate', 'Counts']
)
display(df)

## Estimating the ground state energy with iterative quantum phase estimation

Kitaev-style iterative quantum phase estimation (iQPE) estimates an eigenphase of the time-evolution operator $U = e^{-iHt}$ using one ancilla qubit and a sequence of controlled-$U^{2^k}$ applications.

Each iteration measures one bit of the phase (from most-significant to least-significant) and uses phase feedback to refine the estimate.

The Hückel Hamiltonian must be mapped to a qubit Hamiltonian that can be measured on a quantum computer.
The Jordan-Wigner transformation is a popular mapping that is used in this example.

In [ ]:
# Prepare the qubit-mapped Hamiltonian
qubit_mapper = create("qubit_mapper", algorithm_name="qiskit", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)
print("Qubit Hamiltonian:\n", qubit_hamiltonian.get_summary())

In [ ]:
# Set up parameters for iQPE
from utils.qpe_utils import compute_evolution_time

M_PRECISION = 10
SHOTS_PER_BIT = 3
SIMULATOR_SEED = 42

# For model Hamiltonians (no core energy), use the base time pi/||H|| without refinement
evolution_time = compute_evolution_time(qubit_hamiltonian, num_bits=M_PRECISION, solve_hamiltonian=False)
print(f"Proposed evolution time: {evolution_time:.4f} Hartree^-1")

The circuit for iQPE consists of initial trial state preparation followed by multiple controlled time-evolution operations. This cell visualizes one iteration of the iQPE circuit using built-in `qdk` visualization tools.

In [ ]:
# Use factory methods to create the iQPE algorithm components
evolution_builder = create("time_evolution_builder", "trotter", order=2)
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")
iqpe = create(
    "phase_estimation",
    "iterative",
    num_bits=M_PRECISION,
    evolution_time=evolution_time,
    shots_per_bit=SHOTS_PER_BIT,
)

# Generate the iQPE iteration circuit for a specific iteration (3rd from last)
iqpe_iter_circuit = iqpe.create_iteration_circuit(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
    iteration=M_PRECISION - 3,
    total_iterations=M_PRECISION,
)

# Visualize the iQPE iteration circuit
display(Circuit(iqpe_iter_circuit.get_qsharp_circuit()))

This real-time example performs a single-trial iQPE run on the `qdk` full state simulator.

In [ ]:
# Execute the iQPE algorithm for a single trial
circuit_executor = create("circuit_executor", "qdk_full_state_simulator", seed=SIMULATOR_SEED)
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=circuit_executor,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)

# Summarize the QPE results
estimated_energy = result.raw_energy + hamiltonian.get_core_energy()
estimated_error = abs(estimated_energy - e_exact)
print(f"QPE Results for {M_PRECISION}-bit precision:")
print(f"Reference CASCI energy: {e_exact:.6f} Hartree")
print(f"Total energy from phase estimation: {estimated_energy:.6f} Hartree")
print(f"Energy difference with CASCI energy: {estimated_error:.3e} Hartree")

The above cell used a single trial for a real-time example. However, iQPE generally requires multiple trials to establish confidence in the resulting estimate. The following cell performs a multi-trial iQPE run with high precision using the same simulator.

In [ ]:
NUM_TRIALS = 20
RESULTS_DIR = Path(
    f"results_iqpe_huckel/precision_{M_PRECISION}/time_{round(evolution_time, 12)}"
)

# Run iQPE if results do not already exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for trial in range(NUM_TRIALS):
    trial_seed = SIMULATOR_SEED + trial
    json_path = RESULTS_DIR / f"iqpe_result_{trial_seed}.qpe_result.json"
    if not json_path.exists():
        print(f"Running trial {trial + 1} of {NUM_TRIALS}...")
        executor = create(
            "circuit_executor",
            "qdk_full_state_simulator",
            type="cpu",
            seed=trial_seed,
        )
        result = iqpe.run(
            state_preparation=state_prep_circuit,
            qubit_hamiltonian=qubit_hamiltonian,
            circuit_executor=executor,
            evolution_builder=evolution_builder,
            circuit_mapper=circuit_mapper,
        )
        result.to_json_file(json_path)

For a system with noise or an imperfect trial state, multiple trials of iQPE are needed to obtain a reliable estimate of the ground state energy. This estimate is typically taken as the most frequently observed energy from multiple trials ("majority vote").

In [ ]:
from qdk_chemistry.data import QpeResult

# Load the results
results_loaded = []
for json_file in RESULTS_DIR.glob("*qpe_result.json"):
    result = QpeResult.from_json_file(json_file)
    results_loaded.append(result)

# Count the energy values
energy_counts = Counter(
    [
        result.raw_energy + hamiltonian.get_core_energy()
        for result in results_loaded
    ]
)
print(f"QPE Results of {M_PRECISION} bit precision from {NUM_TRIALS} trials:")
display(pd.DataFrame(energy_counts.items(), columns=['Energy (Hartree)', 'Counts']))

# Use the most frequently observed energy across all trials as the overall estimate
estimated_energy, _ = energy_counts.most_common(1)[0]

The iQPE energy estimate accuracy is useful for benchmarking the impact of precision, evolution time, and other parameters on the final result. The following cell summarizes energy errors from the multiple trials.

In [ ]:
# Print summary of results
print(f"Reference CASCI energy: {e_exact:.6f} Hartree")
print(f"Total energy from phase estimation: {estimated_energy:.6f} Hartree")
print(f"Energy difference with CASCI energy: {abs(estimated_energy - e_exact):.3e} Hartree")

# Summarize the energy errors
energy_errors = {
    qpe_e - e_exact: count
    for qpe_e, count in sorted(energy_counts.items())
}

# Plot distribution of energy differences
print("Energy difference (Hartree) distribution:")
display(
    Histogram(
        bar_values={f"{err:.3e}": count for err, count in energy_errors.items()}
    )
)